In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np

base_path = '/content/drive/My Drive/MIMIC/HEART/'

df = pd.read_csv(base_path + 'final_heart_clinical_master4.csv')

In [3]:
df.isnull().sum()

,0
subject_id,0
hadm_id,0
icd_code,0
visit_procedure_count,0
anchor_age,0
...,...
vital_mean_systolic_bp,0
vital_mean_diastolic_bp,0
vital_mean_spo2,0
vital_mean_temp,0


In [4]:


# 2. Clean the icd column to ensure consistent matching
df['icd_clean'] = df['icd_code'].astype(str).str.strip()

# 3. Create a comprehensive clinical mapping dictionary for your dataset's codes
icd_text_mapping = {
    # --- Major ICD-9 Cardiovascular & Renal Codes ---
    '40391': 'Hypertensive chronic kidney disease, stage 5 or end-stage renal disease',
    '4019': 'Unspecified essential hypertension',
    '42833': 'Acute on chronic diastolic heart failure',
    '4280': 'Congestive heart failure, unspecified',
    '41401': 'Coronary atherosclerosis of native coronary artery',
    '41071': 'Subendocardial infarction, initial episode of care',
    '412': 'Old myocardial infarction',
    '42731': 'Atrial fibrillation',
    '4241': 'Aortic valve disorders',
    '4240': 'Mitral valve disorders',

    # --- Major ICD-10 Cardiovascular Codes ("I-Codes") ---
    'I10': 'Essential (primary) hypertension',
    'I110': 'Hypertensive heart disease with heart failure',
    'I120': 'Hypertensive chronic kidney disease with stage 5 or end-stage renal disease',
    'I214': 'Acute subendocardial myocardial infarction',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'I480': 'Paroxysmal atrial fibrillation',
    'I481': 'Persistent atrial fibrillation',
    'I482': 'Chronic atrial fibrillation',
    'I509': 'Heart failure, unspecified',
    'I5033': 'Acute on chronic diastolic (congestive) heart failure',
    'I5023': 'Acute on chronic systolic (congestive) heart failure'
}

# 4. Apply the mapping to create a new text column
# If a rare code isn't in our manual dictionary, it labels it generically by its system
def get_icd_description(code):
    if code in icd_text_mapping:
        return icd_text_mapping[code]
    elif code.startswith('I'):
        return f"ICD-10 Circulatory/Cardiac Condition (Code: {code})"
    else:
        return f"ICD-9 Cardiovascular Condition (Code: {code})"

df['icd_description'] = df['icd_clean'].apply(get_icd_description)

# 5. Clean up and view your new structured data
# Drop the temporary column and reorder so description sits right next to the code
df = df.drop(columns=['icd_clean'])
cols = list(df.columns)
# Move 'icd_description' next to 'icd_code'
icd_idx = cols.index('icd_code')
cols.insert(icd_idx + 1, cols.pop(cols.index('icd_description')))
df = df[cols]

# Save the newly transformed dataset for your research files
df.to_csv('final_heart_clinical_master_with_text.csv', index=False)

print("Transformation Complete! Sample of your new text descriptions:")
print(df[['subject_id', 'hadm_id', 'icd_code', 'icd_description']].head(10))

Transformation Complete! Sample of your new text descriptions:
   subject_id   hadm_id icd_code  \
0    10000980  26913865    41071   
1    10000980  26913865    42823   
2    10000980  26913865    41412   
3    10000980  26913865     4240   
4    10000980  26913865     4280   
5    10000980  26913865    41401   
6    10000980  26913865    40390   
7    10000980  26913865      412   
8    10002013  24760295    41071   
9    10002013  24760295    42832   

                                     icd_description  
0  Subendocardial infarction, initial episode of ...  
1       ICD-9 Cardiovascular Condition (Code: 42823)  
2       ICD-9 Cardiovascular Condition (Code: 41412)  
3                             Mitral valve disorders  
4              Congestive heart failure, unspecified  
5  Coronary atherosclerosis of native coronary ar...  
6       ICD-9 Cardiovascular Condition (Code: 40390)  
7                          Old myocardial infarction  
8  Subendocardial infarction, initial episode

In [5]:
df.sample(6)

,subject_id,hadm_id,icd_code,icd_description,visit_procedure_count,anchor_age,hospital_expire_flag,is_deceased,report_1st,report_1st_degree,...,title_transluminal,title_transluminal_coronary,title_using,title_using_catheters,vital_mean_heart_rate,vital_mean_systolic_bp,vital_mean_diastolic_bp,vital_mean_spo2,vital_mean_temp,is_male
5312,13011899,25547058,41401,Coronary atherosclerosis of native coronary ar...,2.0,65,0.0,0,0.0,0.0,...,0.000000,0.000000,0.231623,0.289308,79.584507,112.846154,60.906250,96.470588,98.105556,0
16075,19093103,25095647,4019,Unspecified essential hypertension,7.0,51,0.0,0,0.0,0.0,...,0.216186,0.216186,0.000000,0.000000,79.584507,112.846154,60.906250,96.470588,98.105556,0
15145,18533354,26700619,4257,ICD-9 Cardiovascular Condition (Code: 4257),3.0,81,0.0,1,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,74.755725,90.836957,44.347826,92.515748,97.832143,0
7666,14254388,29691672,41401,Coronary atherosclerosis of native coronary ar...,8.0,58,0.0,0,0.0,0.0,...,0.202249,0.202249,0.126569,0.000000,79.584507,112.846154,60.906250,96.470588,98.105556,1
13150,17414813,20766522,4280,"Congestive heart failure, unspecified",3.0,77,0.0,1,0.0,0.0,...,0.000000,0.000000,0.223690,0.279399,79.584507,112.846154,60.906250,96.470588,98.105556,1
2441,11427507,27594123,I2582,ICD-10 Circulatory/Cardiac Condition (Code: I2...,1.0,62,0.0,0,0.0,0.0,...,0.000000,0.000000,0.721096,0.000000,79.584507,112.846154,60.906250,96.470588,98.105556,0


In [6]:
y = df['is_deceased'].astype(int)
groups = df['subject_id']

In [7]:
cols_to_exclude = ['subject_id', 'hadm_id', 'icd_code', 'hospital_expire_flag', 'is_deceased']
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

In [8]:
# 4. ENCODING: One-Hot Encode your text description column
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')

In [ ]:
# new method

In [13]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_encoded, y, groups=groups))

X_train, X_test = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [14]:
# new method

In [17]:
# 6. SCALING: Identify continuous columns that need scaling
from sklearn.preprocessing import StandardScaler
continuous_cols = ['anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
                   'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
                   'vital_mean_spo2', 'vital_mean_temp']

scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print(f"Data processing complete!")
print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")

Data processing complete!
Train Shape: (14136, 599) | Test Shape: (3619, 599)


In [20]:
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators=250, max_depth=4, learning_rate=0.04, scale_pos_weight=1.3, random_state=42)
model.fit(X_train, y_train)

# 8. PATIENT-LEVEL EVALUATION
y_prob = model.predict_proba(X_test)[:, 1]
test_summary = pd.DataFrame({
    'hadm_id': df.iloc[test_idx]['hadm_id'],
    'y_true': y_test,
    'y_prob': y_prob
})

In [21]:
from sklearn.metrics import classification_report, roc_auc_score
# Average risk per hospital admission
patient_results = test_summary.groupby('hadm_id').mean()
patient_preds = (patient_results['y_prob'] >= 0.5).astype(int)

print("\n--- Final Structured Pipeline Patient Results ---")
print(classification_report(patient_results['y_true'], patient_preds))
print(f"Patient-Level ROC-AUC Score: {roc_auc_score(patient_results['y_true'], patient_results['y_prob']):.4f}")


--- Final Structured Pipeline Patient Results ---
              precision    recall  f1-score   support

         0.0       0.75      0.74      0.74       432
         1.0       0.57      0.59      0.58       258

    accuracy                           0.68       690
   macro avg       0.66      0.66      0.66       690
weighted avg       0.68      0.68      0.68       690

Patient-Level ROC-AUC Score: 0.7365


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load your master dataset
df = pd.read_csv('final_heart_clinical_master_with_text.csv')

# 2. Separate Target and Group Identifiers
y = df['is_deceased'].astype(int)
groups = df['subject_id']  # Grouping identifier to prevent patient data leakage

# 3. Handle Features and One-Hot Encoding
cols_to_exclude = ['subject_id', 'hadm_id', 'icd_code', 'hospital_expire_flag', 'is_deceased']
X_raw = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])
X_encoded = pd.get_dummies(X_raw, columns=['icd_description'], prefix='Dx', prefix_sep='_')

# 4. Initialize GroupKFold (5 Folds is standard for research)
gkf = GroupKFold(n_splits=5)

# List of continuous columns that need scaling
continuous_cols = ['anchor_age', 'visit_procedure_count', 'vital_mean_heart_rate',
                   'vital_mean_systolic_bp', 'vital_mean_diastolic_bp',
                   'vital_mean_spo2', 'vital_mean_temp']

# Storage array to collect predictions across all folds
oof_predictions = pd.DataFrame()

print(f"Beginning 5-Fold GroupKFold Cross-Validation...\n")

# 5. Run the Cross-Validation Loop
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_encoded, y, groups=groups), 1):
    print(f"--- Training Fold {fold} ---")

    # Split features, targets, and tracking columns for this fold
    X_train, X_val = X_encoded.iloc[train_idx].copy(), X_encoded.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    hadm_val = df.iloc[val_idx]['hadm_id']

    # SCALE features within the loop to keep fold splits pure
    scaler = StandardScaler()
    X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
    X_val[continuous_cols] = scaler.transform(X_val[continuous_cols])

    # Train the XGBoost model for this fold
    model = XGBClassifier(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.04,
        scale_pos_weight=1.3,
        random_state=42
    )
    model.fit(X_train, y_train)

    # Predict probabilities for the validation fold rows
    y_prob_val = model.predict_proba(X_val)[:, 1]

    # Store validation outputs for this specific fold
    fold_results = pd.DataFrame({
        'hadm_id': hadm_val,
        'y_true': y_val,
        'y_prob': y_prob_val
    })
    oof_predictions = pd.concat([oof_predictions, fold_results], axis=0)

# ========================================================
# 6. GLOBAL PATIENT-LEVEL EVALUATION (Across All 5 Folds)
# ========================================================
print("\n========================================================")
print("     FINAL PATIENT-LEVEL GROUPKFOLD EVALUATION         ")
print("========================================================")

# Average all row probabilities down to unique hospital stays (hadm_id)
final_patient_results = oof_predictions.groupby('hadm_id').mean()
final_patient_preds = (final_patient_results['y_prob'] >= 0.5).astype(int)

# Generate ultimate validation metrics
print(classification_report(final_patient_results['y_true'], final_patient_preds))
print(f"Overall Cross-Validated Patient-Level ROC-AUC Score: {roc_auc_score(final_patient_results['y_true'], final_patient_results['y_prob']):.4f}")

Beginning 5-Fold GroupKFold Cross-Validation...

--- Training Fold 1 ---
--- Training Fold 2 ---
--- Training Fold 3 ---
--- Training Fold 4 ---
--- Training Fold 5 ---

     FINAL PATIENT-LEVEL GROUPKFOLD EVALUATION         
              precision    recall  f1-score   support

         0.0       0.76      0.74      0.75      2181
         1.0       0.58      0.60      0.59      1278

    accuracy                           0.69      3459
   macro avg       0.67      0.67      0.67      3459
weighted avg       0.69      0.69      0.69      3459

Overall Cross-Validated Patient-Level ROC-AUC Score: 0.7426
